In [12]:
import numpy as np
import pandas as pd
import sys,time,json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold,GroupKFold,GridSearchCV,StratifiedGroupKFold, RandomizedSearchCV
from sklearn.utils import shuffle
from xgboost import XGBClassifier
from scipy.stats import randint
pd.set_option('display.max_columns', None)

In [13]:
subject_info_raw = pd.read_csv('/Users/eric/repos/aud/data/subject_info.csv')
subject_info = subject_info_raw.query("responsiveness>=0.25").copy(deep=True)
print(subject_info.shape)
subject_info.head()

(148, 5)


,subid,mema_count,last_morning_ema_day,first_morning_ema_day,responsiveness
0,1,64,89,0,0.711111
1,2,77,84,0,0.905882
2,3,87,89,0,0.966667
3,5,90,91,0,0.988889
4,6,79,86,0,0.908046


In [14]:
labels = pd.read_csv('/Users/eric/repos/aud/data/ml_labels_full_width.csv')
labels.head()

,subid,day,day_of_week_num_0,day_of_week_num_1,day_of_week_num_2,day_of_week_num_3,day_of_week_num_4,day_of_week_num_5,day_of_week_num_6,latest_ema_2,latest_ema_3,latest_ema_4,latest_ema_5,latest_ema_6,latest_ema_7,latest_ema_8,latest_ema_9,latest_ema_10,srm_ema_2,srm_ema_3,srm_ema_4,srm_ema_5,srm_ema_6,srm_ema_7,srm_ema_8,srm_ema_9,srm_ema_10,lrm_ema_2,lrm_ema_3,lrm_ema_4,lrm_ema_5,lrm_ema_6,lrm_ema_7,lrm_ema_8,lrm_ema_9,lrm_ema_10,recent_lapse_1,recent_lapse_3,recent_lapse_5,lapse_w0,lapse_w1,lapse_w3,lapse_w7,train_width
0,1,0,0,0,0,1,0,0,0,0.333333,1.000000,1.000000,0.666667,0.5,0.7,0.4,0.4,0.3,0.333333,1.000000,1.000000,0.666667,0.500000,0.700000,0.400000,0.4,0.3,0.333333,1.000000,1.000000,0.666667,0.50,0.700000,0.400000,0.400,0.3,False,False,False,0,0.0,0.0,0.0,90
1,1,1,0,0,0,0,1,0,0,0.166667,0.333333,0.333333,0.666667,0.5,0.2,0.4,0.6,0.3,0.250000,0.666667,0.666667,0.666667,0.500000,0.450000,0.400000,0.5,0.3,0.250000,0.666667,0.666667,0.666667,0.50,0.450000,0.400000,0.500,0.3,False,False,False,0,0.0,0.0,0.0,90
2,1,2,0,0,0,0,0,1,0,0.166667,0.333333,0.333333,0.666667,0.5,0.2,0.4,0.6,0.3,0.250000,0.666667,0.666667,0.666667,0.500000,0.450000,0.400000,0.5,0.3,0.250000,0.666667,0.666667,0.666667,0.50,0.450000,0.400000,0.500,0.3,False,False,False,0,0.0,0.0,0.0,90
3,1,3,0,0,0,0,0,0,1,0.166667,0.000000,0.000000,0.416667,0.5,0.2,0.3,0.5,0.3,0.222222,0.444444,0.444444,0.583333,0.500000,0.366667,0.366667,0.5,0.3,0.222222,0.444444,0.444444,0.583333,0.50,0.366667,0.366667,0.500,0.3,False,False,False,0,0.0,0.0,0.0,90
4,1,4,1,0,0,0,0,0,0,0.166667,0.083333,0.333333,0.666667,0.7,0.7,0.6,0.4,0.3,0.166667,0.138889,0.222222,0.583333,0.566667,0.366667,0.433333,0.5,0.3,0.208333,0.354167,0.416667,0.604167,0.55,0.450000,0.425000,0.475,0.3,False,False,False,0,0.0,0.0,0.0,90


In [15]:
ref = labels[['subid','day','lapse_w0']]
ref.rename(mapper={"lapse_w0":"lapse"},inplace=True,axis=1)
print("Overall lapse fraction", np.sum(ref.lapse)/ref.shape[0])
ref.head()

Overall lapse fraction 0.07548520521794463


/var/folders/yv/fz_ldjls7497yzp899r2yzw80000gn/T/ipykernel_6746/664960605.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ref.rename(mapper={"lapse_w0":"lapse"},inplace=True,axis=1)


,subid,day,lapse
0,1,0,0
1,1,1,0
2,1,2,0
3,1,3,0
4,1,4,0


In [17]:
# Multi-fold testing
# Desired number of folds per repeat
splits=5
# Desired random states for each repeat
#random_states = [1]
random_states = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]

# Dictionaries for storing information
fold_dict = {}
ssm_dict = {}

# Get relevant X/Y/group information
X = ref[['subid','day']]
Y = ref[['lapse']]
groups = ref.subid

# Loop over the random states (i.e. loop over repeats)
for random_state in random_states:
    print("Random state: {}".format(random_state))
    # Set up sklearn object for generating folds
    sgkf = StratifiedGroupKFold(n_splits=splits,shuffle=True,random_state=random_state)
    test_list = []
    
    # Iterate over the train/test splits within this repeat
    for i,(train_index,test_index) in enumerate(sgkf.split(X,Y,groups=groups)):
        # Gather information about this split
        train_count = len(train_index)
        test_count = len(test_index)
        train_lapse = np.sum(Y.loc[train_index,"lapse"])
        train_prop = train_lapse/train_count
        test_lapse = np.sum(Y.loc[test_index,"lapse"])
        test_prop = test_lapse/test_count
        # Display split properties
        print("Fold {}, train count {}, test count {}, train_lapse {}, train lapse prop. {:.3f}, test_lapse {}, test lapse prop. {:.3f}"\
            .format(i,train_count,test_count,train_lapse,train_prop,test_lapse,test_prop))
        
        # Get subids included in train/test splits
        test_subids_iter = X.loc[test_index].subid.unique()
        train_subids_iter = X.loc[train_index].subid.unique()
        # Print the exact subject split info
        #print(test_subids_iter.tolist(),len(test_subids_iter))

        # Sanity checking that subids don't appear more than once
        if any([val in test_list for val in test_subids_iter]):
            print("error!")
            break
        else:
            test_list.extend(test_subids_iter.tolist())

        # Record information about the split
        fold_id = "split{}_rs{}_fold{}_id0".format(str(splits),str(random_state),str(i))
        fold_dict[fold_id]={'train':train_subids_iter.tolist(),'test':test_subids_iter.tolist(),'train_lapse_prop':train_prop,'test_lapse_prop':test_prop,'fold':i,'random_state':random_state}

        for id in test_subids_iter:
            ssm_key = "split{}_rs{}_fold{}_id{}".format(str(splits),str(random_state),str(i),str(id))
            ssm_dict[ssm_key] = {'train':train_subids_iter.tolist(),'test':int(id),'train_lapse_prop':train_prop,'test_lapse_prop':test_prop,'fold':i,'random_state':random_state}

ml_json = json.dumps(fold_dict, indent=4)
#print(ml_json)
with open('ml_fold_reference.json','w') as f:
    json.dump(fold_dict,indent=4,fp=f)

ssm_json = json.dumps(ssm_dict, indent=4)
#print(ssm_json)
with open('/Users/eric/repos/aud/data/ssm_fold_reference.json','w') as f:
    json.dump(ssm_dict,indent=4,fp=f)

# ML writing
ml_path = 'subid_ml_list_data_eff.txt'
f = open(ml_path,'w')
f.close()
f = open(ml_path,'a')
key_list = list(fold_dict.keys())
for key in key_list:
    if(key==key_list[-1]):
        f.write(key)
    else:
        f.write("{}\n".format(key))
f.close()

# SSM writing
ssm_path = '/Users/eric/repos/aud/chtc/subid_list_data_eff.txt'
f = open(ssm_path,'w')
f.close()
f = open(ssm_path,'a')
key_list = list(ssm_dict.keys())
for key in key_list:
    if(key==key_list[-1]):
        f.write("{} {}".format(key,'MAP_mle'))
    else:
        f.write("{} {}\n".format(key,'MAP_mle'))
f.close()

Random state: 1
Fold 0, train count 9940, test count 2632, train_lapse 817, train lapse prop. 0.082, test_lapse 132, test lapse prop. 0.050
Fold 1, train count 10089, test count 2483, train_lapse 737, train lapse prop. 0.073, test_lapse 212, test lapse prop. 0.085
Fold 2, train count 9833, test count 2739, train_lapse 743, train lapse prop. 0.076, test_lapse 206, test lapse prop. 0.075
Fold 3, train count 10283, test count 2289, train_lapse 761, train lapse prop. 0.074, test_lapse 188, test lapse prop. 0.082
Fold 4, train count 10143, test count 2429, train_lapse 738, train lapse prop. 0.073, test_lapse 211, test lapse prop. 0.087
Random state: 2
Fold 0, train count 9853, test count 2719, train_lapse 794, train lapse prop. 0.081, test_lapse 155, test lapse prop. 0.057
Fold 1, train count 10119, test count 2453, train_lapse 787, train lapse prop. 0.078, test_lapse 162, test lapse prop. 0.066
Fold 2, train count 10126, test count 2446, train_lapse 740, train lapse prop. 0.073, test_lapse

In [5]:
splits=5
random_state=1
sgkf = StratifiedGroupKFold(n_splits=splits,shuffle=True,random_state=random_state)
X = ref[['subid','day']]
Y = ref[['lapse']]
groups = ref.subid
test_list = []
train_groups = {}
test_groups = {}
#fold_df_list = []
fold_dict = {}
ssm_dict = {}
#ssm_df_list = []
for i,(train_index,test_index) in enumerate(sgkf.split(X,Y,groups=groups)):
    train_count = len(train_index)
    test_count = len(test_index)
    train_lapse = np.sum(Y.loc[train_index,"lapse"])
    train_prop = train_lapse/train_count
    test_lapse = np.sum(Y.loc[test_index,"lapse"])
    test_prop = test_lapse/test_count
    print("Fold {}, train count {}, test count {}, train_lapse {}, train lapse prop. {:.3f}, test_lapse {}, test lapse prop. {:.3f}"\
          .format(i,train_count,test_count,train_lapse,train_prop,test_lapse,test_prop))
    #print(X.loc[train_index].subid.unique())
    test_subids_iter = X.loc[test_index].subid.unique()
    train_subids_iter = X.loc[train_index].subid.unique()
    print(test_subids_iter.tolist(),len(test_subids_iter))
    if any([val in test_list for val in test_subids_iter]):
        print("error!")
        break
    else:
        test_list.extend(test_subids_iter.tolist())
    test_groups[i]=test_subids_iter
    #print(test_subids_iter)
    train_groups[i]=train_subids_iter

    fold_id = "split{}_rs{}_fold{}_id0".format(str(splits),str(random_state),str(i))
    fold_dict[fold_id]={'train':train_subids_iter.tolist(),'test':test_subids_iter.tolist(),'train_lapse_prop':train_prop,'test_lapse_prop':test_prop,'fold':i,'random_state':random_state}

    for id in test_subids_iter:
        ssm_key = "split{}_rs{}_fold{}_id{}".format(str(splits),str(random_state),str(i),str(id))
        ssm_dict[ssm_key] = {'train':train_subids_iter.tolist(),'test':int(id),'train_lapse_prop':train_prop,'test_lapse_prop':test_prop,'fold':i,'random_state':random_state}

ml_json = json.dumps(fold_dict, indent=4)
print(ml_json)
with open('ml_fold_reference.json','w') as f:
    json.dump(fold_dict,indent=4,fp=f)

ssm_json = json.dumps(ssm_dict, indent=4)
print(ssm_json)
with open('/Users/eric/repos/aud/data/ssm_fold_reference.json','w') as f:
    json.dump(ssm_dict,indent=4,fp=f)

# ML writing
ml_path = 'subid_ml_list_data_eff.txt'
f = open(ml_path,'w')
f.close()
f = open(ml_path,'a')
key_list = list(fold_dict.keys())
for key in key_list:
    if(key==key_list[-1]):
        f.write(key)
    else:
        f.write("{}\n".format(key))
f.close()

# SSM writing
ssm_path = '/Users/eric/repos/aud/chtc/subid_list_data_eff.txt'
f = open(ssm_path,'w')
f.close()
f = open(ssm_path,'a')
key_list = list(ssm_dict.keys())
for key in key_list:
    if(key==key_list[-1]):
        f.write("{} {}".format(key,'MAP_mle'))
    else:
        f.write("{} {}\n".format(key,'MAP_mle'))
f.close()

Fold 0, train count 11194, test count 1378, train_lapse 831, train lapse prop. 0.074, test_lapse 118, test lapse prop. 0.086
[20, 28, 30, 33, 38, 44, 54, 58, 97, 109, 135, 188, 192, 200, 231, 241] 16
Fold 1, train count 11298, test count 1274, train_lapse 858, train lapse prop. 0.076, test_lapse 91, test lapse prop. 0.071
[15, 21, 31, 32, 43, 51, 63, 78, 84, 110, 116, 213, 214, 243, 262] 15
Fold 2, train count 11365, test count 1207, train_lapse 913, train lapse prop. 0.080, test_lapse 36, test lapse prop. 0.030
[18, 19, 34, 74, 76, 81, 118, 136, 156, 196, 215, 238, 240, 264] 14
Fold 3, train count 11258, test count 1314, train_lapse 893, train lapse prop. 0.079, test_lapse 56, test lapse prop. 0.043
[11, 29, 39, 88, 143, 149, 158, 163, 166, 169, 183, 203, 207, 222, 245] 15
Fold 4, train count 11442, test count 1130, train_lapse 828, train lapse prop. 0.072, test_lapse 121, test lapse prop. 0.107
[5, 6, 37, 48, 52, 64, 90, 92, 117, 172, 175, 189, 212, 223] 14
Fold 5, train count 11351,

In [15]:
ref.shape

(12572, 3)

In [41]:
# LOO case

X = ref[['subid','day']]
Y = ref[['lapse']]

test_list = []
train_groups = {}
test_groups = {}

fold_dict = {}
ssm_dict = {}

included_subjects = np.unique(ref.subid)
for i, subject in enumerate(included_subjects):
    train_index = ref.query("subid!=@subject").index
    test_index = ref.query("subid==@subject").index
    train_count = len(train_index)
    test_count = len(test_index)
    train_lapse = np.sum(Y.loc[train_index,"lapse"])
    train_prop = train_lapse/train_count
    test_lapse = np.sum(Y.loc[test_index,"lapse"])
    test_prop = test_lapse/test_count
    print("Fold {}, train count {}, test count {}, train_lapse {}, train lapse prop. {:.3f}, test_lapse {}, test lapse prop. {:.3f}"\
          .format(i,train_count,test_count,train_lapse,train_prop,test_lapse,test_prop))
    test_subids_iter = X.loc[test_index].subid.unique()
    train_subids_iter = X.loc[train_index].subid.unique()
    print(test_subids_iter.tolist(),len(test_subids_iter))
    if any([val in test_list for val in test_subids_iter]):
        print("error!")
        break
    else:
        test_list.extend(test_subids_iter.tolist())
    test_groups[i]=test_subids_iter
    #print(test_subids_iter)
    train_groups[i]=train_subids_iter

    fold_id = "splitLOO0_rsLOO0_foldLOO0_id{}".format(str(subject))
    fold_dict[fold_id]={'train':train_subids_iter.tolist(),'test':test_subids_iter.tolist(),'train_lapse_prop':train_prop,'test_lapse_prop':test_prop,'fold':'LOO','random_state':'LOO'}
    ssm_dict[fold_id] = {'train':train_subids_iter.tolist(),'test':int(subject),'train_lapse_prop':train_prop,'test_lapse_prop':test_prop,'fold':'LOO','random_state':'LOO'}

ml_json = json.dumps(fold_dict, indent=4)
print(ml_json)
with open('ml_fold_reference.json','w') as f:
    json.dump(fold_dict,indent=4,fp=f)

ssm_json = json.dumps(ssm_dict, indent=4)
print(ssm_json)
with open('/Users/eric/repos/aud/data/ssm_fold_reference.json','w') as f:
    json.dump(ssm_dict,indent=4,fp=f)

# ML writing
ml_path = 'subid_ml_list_data_eff.txt'
f = open(ml_path,'w')
f.close()
f = open(ml_path,'a')
key_list = list(fold_dict.keys())
for key in key_list:
    if(key==key_list[-1]):
        f.write(key)
    else:
        f.write("{}\n".format(key))
f.close()

# SSM writing
ssm_path = '/Users/eric/repos/aud/chtc/subid_list_data_eff.txt'
f = open(ssm_path,'w')
f.close()
f = open(ssm_path,'a')
key_list = list(ssm_dict.keys())
for key in key_list:
    if(key==key_list[-1]):
        f.write("{} {}".format(key,'MAP_mle'))
    else:
        f.write("{} {}\n".format(key,'MAP_mle'))
f.close()

Fold 0, train count 12482, test count 90, train_lapse 949, train lapse prop. 0.076, test_lapse 0, test lapse prop. 0.000
[1] 1
Fold 1, train count 12487, test count 85, train_lapse 944, train lapse prop. 0.076, test_lapse 5, test lapse prop. 0.059
[2] 1
Fold 2, train count 12482, test count 90, train_lapse 940, train lapse prop. 0.075, test_lapse 9, test lapse prop. 0.100
[3] 1
Fold 3, train count 12482, test count 90, train_lapse 949, train lapse prop. 0.076, test_lapse 0, test lapse prop. 0.000
[5] 1
Fold 4, train count 12485, test count 87, train_lapse 949, train lapse prop. 0.076, test_lapse 0, test lapse prop. 0.000
[6] 1
Fold 5, train count 12482, test count 90, train_lapse 932, train lapse prop. 0.075, test_lapse 17, test lapse prop. 0.189
[7] 1
Fold 6, train count 12482, test count 90, train_lapse 947, train lapse prop. 0.076, test_lapse 2, test lapse prop. 0.022
[9] 1
Fold 7, train count 12483, test count 89, train_lapse 934, train lapse prop. 0.075, test_lapse 15, test lapse 